# Project 100 — Local AI Ops Mini-Platform (Capstone)
## The Grand Finale: Chat + RAG + Tools + Evals — All Combined

**Stack:** LangChain · LangGraph · Ollama · ChromaDB · Pydantic · Jupyter

This capstone project combines techniques from all 99 previous projects into a single
integrated platform: conversational AI, RAG retrieval, tool use, structured output,
evaluation, and observability — all running locally.

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb pydantic

## Module 1 — Core LLM & Embeddings

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
import json, time, shutil
from pathlib import Path
from datetime import datetime

# Core models
llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("✓ Module 1: Core LLM & Embeddings initialized")

## Module 2 — Knowledge Base (RAG)

In [ ]:
# Build a knowledge base from the project's own documentation
knowledge_docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications. "
        "It provides chains, agents, and retrieval components. Key abstractions include "
        "ChatModels, Prompts, OutputParsers, and Retrievers.", metadata={"topic": "langchain"}),
    Document(page_content="Ollama runs LLMs locally. Supported models include Llama 3, Qwen, "
        "Mistral, Phi-3, and Gemma. It provides an OpenAI-compatible API at localhost:11434. "
        "Models are pulled with 'ollama pull model_name'.", metadata={"topic": "ollama"}),
    Document(page_content="ChromaDB is an open-source vector database for AI applications. "
        "It stores embeddings and supports similarity search. It can run in-memory or "
        "persistently. Key operations: add, query, delete.", metadata={"topic": "chromadb"}),
    Document(page_content="LangGraph extends LangChain with stateful, multi-step workflows. "
        "It uses StateGraph with typed state dictionaries. Nodes are functions, edges define "
        "flow. Supports conditional routing and human-in-the-loop.", metadata={"topic": "langgraph"}),
    Document(page_content="RAG (Retrieval-Augmented Generation) combines search with generation. "
        "Steps: 1) Index documents as embeddings, 2) Retrieve relevant chunks for a query, "
        "3) Generate answers grounded in retrieved context.", metadata={"topic": "rag"}),
    Document(page_content="CrewAI enables multi-agent collaboration. Define Agents with roles "
        "and backstories, Tasks with descriptions, and Crews that orchestrate execution. "
        "Supports sequential and hierarchical processes.", metadata={"topic": "crewai"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(knowledge_docs)

shutil.rmtree("chroma_capstone", ignore_errors=True)
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_capstone")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✓ Module 2: Knowledge base built — {len(chunks)} chunks indexed")

## Module 3 — Tool Registry

In [ ]:
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the AI knowledge base for information."""
    docs = retriever.invoke(query)
    return "\n".join([f"[{d.metadata.get('topic','?')}] {d.page_content}" for d in docs])

@tool
def analyze_code(code: str) -> str:
    """Analyze Python code for issues and improvements."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Analyze this code briefly. List any bugs, security issues, or improvements."),
        ("human", "{code}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"code": code})

@tool
def generate_summary(text: str) -> str:
    """Summarize text into 3 key bullet points."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Summarize in exactly 3 bullet points."),
        ("human", "{text}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"text": text})

tools = [search_knowledge_base, analyze_code, generate_summary]
print(f"✓ Module 3: {len(tools)} tools registered: {[t.name for t in tools]}")

## Module 4 — Evaluation Engine

In [ ]:
class EvalResult(BaseModel):
    query: str
    answer: str
    groundedness: float = Field(ge=0, le=1)
    relevance: float = Field(ge=0, le=1)
    quality: float = Field(ge=0, le=1)
    latency_s: float

eval_judge = llm.with_structured_output(EvalResult)

class TraceLog:
    def __init__(self):
        self.entries = []

    def log(self, event, data, duration=0):
        self.entries.append({
            "timestamp": datetime.now().isoformat(),
            "event": event,
            "data_size": len(str(data)),
            "duration_s": round(duration, 3),
        })

    def summary(self):
        return {
            "total_events": len(self.entries),
            "total_duration": sum(e["duration_s"] for e in self.entries),
            "events": [e["event"] for e in self.entries],
        }

trace = TraceLog()
print("✓ Module 4: Evaluation engine & trace logger ready")

## Module 5 — Integrated Pipeline

In [ ]:
def ai_ops_pipeline(query):
    """Full pipeline: retrieve → generate → evaluate → trace."""
    results = {}

    # Step 1: Retrieve
    start = time.time()
    context_docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in context_docs])
    retrieve_time = time.time() - start
    trace.log("retrieve", context, retrieve_time)

    # Step 2: Generate
    start = time.time()
    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer based on the context. Be accurate and cite sources."),
        ("human", "Context:\n{context}\n\nQuestion: {query}")
    ])
    chain = qa_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "query": query})
    generate_time = time.time() - start
    trace.log("generate", answer, generate_time)

    # Step 3: Evaluate
    start = time.time()
    try:
        evaluation = eval_judge.invoke(
            f"Evaluate this Q&A:\nQuery: {query}\nAnswer: {answer}\n"
            f"Context: {context[:300]}"
        )
        eval_scores = {
            "groundedness": evaluation.groundedness,
            "relevance": evaluation.relevance,
            "quality": evaluation.quality,
        }
    except Exception:
        eval_scores = {"groundedness": 0.5, "relevance": 0.5, "quality": 0.5}
    eval_time = time.time() - start
    trace.log("evaluate", eval_scores, eval_time)

    return {
        "query": query,
        "answer": answer,
        "sources": [d.metadata.get("topic") for d in context_docs],
        "scores": eval_scores,
        "timing": {
            "retrieve": round(retrieve_time, 2),
            "generate": round(generate_time, 2),
            "evaluate": round(eval_time, 2),
            "total": round(retrieve_time + generate_time + eval_time, 2),
        },
    }

print("✓ Module 5: Integrated pipeline ready")

## Module 6 — Run the Platform

In [ ]:
queries = [
    "What is RAG and how does it work?",
    "How does LangGraph differ from LangChain?",
    "What models can I run with Ollama?",
    "Explain how ChromaDB stores embeddings",
    "How do CrewAI agents collaborate?",
]

all_results = []
print("LOCAL AI OPS PLATFORM — RUNNING")
print("="*60)

for q in queries:
    result = ai_ops_pipeline(q)
    all_results.append(result)
    print(f"\nQ: {result['query']}")
    print(f"A: {result['answer'][:200]}...")
    print(f"Sources: {result['sources']}")
    print(f"Scores: G={result['scores']['groundedness']:.0%} "
          f"R={result['scores']['relevance']:.0%} "
          f"Q={result['scores']['quality']:.0%}")
    print(f"Timing: {result['timing']['total']:.1f}s "
          f"(retrieve={result['timing']['retrieve']:.1f} "
          f"generate={result['timing']['generate']:.1f} "
          f"eval={result['timing']['evaluate']:.1f})")

## Module 7 — Platform Dashboard

In [ ]:
import pandas as pd

# Performance summary
metrics_df = pd.DataFrame([{
    "query": r["query"][:40],
    "groundedness": r["scores"]["groundedness"],
    "relevance": r["scores"]["relevance"],
    "quality": r["scores"]["quality"],
    "latency": r["timing"]["total"],
} for r in all_results])

print("\n" + "="*60)
print("PLATFORM DASHBOARD")
print("="*60)
print(f"\nQueries processed: {len(all_results)}")
print(f"\nQuality Metrics:")
print(f"  Avg Groundedness: {metrics_df['groundedness'].mean():.0%}")
print(f"  Avg Relevance:    {metrics_df['relevance'].mean():.0%}")
print(f"  Avg Quality:      {metrics_df['quality'].mean():.0%}")
print(f"\nPerformance:")
print(f"  Avg Latency:  {metrics_df['latency'].mean():.1f}s")
print(f"  P95 Latency:  {metrics_df['latency'].quantile(0.95):.1f}s")

print(f"\nTrace Summary:")
trace_summary = trace.summary()
print(f"  Total events: {trace_summary['total_events']}")
print(f"  Total time:   {trace_summary['total_duration']:.1f}s")

print(f"\nDetailed Results:")
print(metrics_df.to_string(index=False))

print(f"\n{'='*60}")
print("🎉 CAPSTONE COMPLETE — All 100 Projects Finished!")
print("="*60)

## Capstone Summary

This project combined ALL major techniques from the 100-project series:

| Module | Technique | From Projects |
|--------|-----------|---------------|
| Core LLM | Ollama + LangChain | 1-10 |
| Knowledge Base | ChromaDB + RAG | 11-30 |
| Tools | Tool registry + execution | 51-60 |
| Evaluation | LLM-as-judge + scoring | 61-70 |
| Observability | Trace logging + metrics | 61-70 |
| Structured Output | Pydantic models | Throughout |
| Pipeline | End-to-end orchestration | 31-40 |

**Congratulations on completing all 100 Local AI Projects!**